# Fine-tune Llama 3.2 for Urdu Drama Scripts (Kaggle)

Manual-token flow: **run Cell 1, paste your HF token, done** — no Kaggle Secrets needed.

**Before you run anything:**
- Settings → Accelerator → **T4 x2** (P100 lacks bfloat16)
- Settings → Internet → **On**
- HF token: https://huggingface.co/settings/tokens (Read+Write scopes)

**Order matters after every kernel restart:** run Cells 1→7 top to bottom. All state (token, working dir) is rebuilt by the cells themselves.

In [ ]:
# Cell 1: Config + manual HF login (paste token when prompted, input is hidden)
import os
from pathlib import Path
from getpass import getpass

# --- Paths (absolute; nothing depends on the notebook's cwd) ---
REPO_URL = 'https://github.com/danielosinor123-gif/llama-3.2-urdu.git'
REPO_DIR = '/kaggle/working/repo'
OUTPUT_DIR = '/kaggle/working/urdu-drama-3.2-qLoRA'
REPORTS_DIR = '/kaggle/working/reports'

# --- Model ---
# NousResearch mirror = identical weights, no gate, runs immediately.
# Switch to 'meta-llama/Llama-3.2-3B-Instruct' once your HF account is granted access.
MODEL_NAME = 'NousResearch/Llama-3.2-3B-Instruct'

CONFIG = {
    'model_name': MODEL_NAME,
    'epochs': 2,
    'batch_size': 2,
    'gradient_accumulation': 8,
    'learning_rate': 2e-4,
    'max_seq_length': 2048,
    'lora_r': 16,
    'lora_alpha': 32,
    'seed': 42,
}

# --- HF credentials: enter once per session ---
os.environ.setdefault('HF_USERNAME', 'danielosinor123-gif')
os.environ.setdefault('HF_REPO', 'urdu-drama-3.2-qLoRA')
if not os.environ.get('HF_TOKEN'):
    os.environ['HF_TOKEN'] = getpass('Paste your HF token (hidden): ')

from huggingface_hub import whoami
try:
    me = whoami(token=os.environ['HF_TOKEN'])
    print('Logged in as:', me.get('name'))
except Exception as e:
    raise SystemExit(f'Token invalid or expired ({e}). Get a new one at https://huggingface.co/settings/tokens')

print('Model:', CONFIG['model_name'])
print('Upload target:', os.environ['HF_USERNAME'] + '/' + os.environ['HF_REPO'])

import torch
if not torch.cuda.is_available():
    raise SystemExit('No GPU. Settings > Accelerator > T4 x2, then restart and run all.')
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# Cell 2: Install what Kaggle does not preinstall (adds trl + metrics packages)
%pip install -q --only-binary=:all: trl bitsandbytes evaluate rouge_score sacrebleu
import transformers, trl, tokenizers
print(f'transformers {transformers.__version__} | trl {trl.__version__} | tokenizers {tokenizers.__version__}')

In [ ]:
# Cell 3: Get the repo (pull if present, re-clone on conflict, clone if fresh)
import os
import shutil
import subprocess
from pathlib import Path

if (Path(REPO_DIR) / '.git').exists():
    print('Repo exists. Pulling latest...')
    r = subprocess.run(['git', '-C', REPO_DIR, 'pull', '--ff-only'], capture_output=True, text=True)
    print(r.stdout, r.stderr)
    if r.returncode != 0:
        print('Pull failed. Re-cloning...')
        shutil.rmtree(REPO_DIR, ignore_errors=True)

if not (Path(REPO_DIR) / '.git').exists():
    print('Cloning the repo...')
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, REPO_DIR], check=True)

SRC = Path(REPO_DIR) / 'src'
SEED_DIR = Path(REPO_DIR) / 'data' / 'seed'
FINAL_DIR = Path(REPO_DIR) / 'data' / 'final'

seed_files = sorted(p.name for p in SEED_DIR.glob('*.jsonl'))
assert seed_files, 'Seed data missing - the clone did not get the latest repo. Re-run this cell.'
print('Seed files:', seed_files)
print('Repo ready at:', REPO_DIR)

In [ ]:
# Cell 4: Build dataset + stress test (training is blocked if any check fails)
import json
import subprocess

steps = [
    ['python', str(SRC / 'data' / 'build_dataset.py'), '--seed_dir', str(SEED_DIR), '--output_dir', str(FINAL_DIR)],
    ['python', str(SRC / 'stress_test.py'), '--data_dir', str(FINAL_DIR), '--seed_dir', str(SEED_DIR)],
]
for cmd in steps:
    print('>>', ' '.join(cmd))
    r = subprocess.run(cmd, capture_output=True, text=True)
    print(r.stdout)
    if r.returncode != 0:
        print(r.stderr)
        raise SystemExit('Step failed: ' + cmd[1] + ' - fix the errors above before training.')

report = json.load(open(FINAL_DIR / 'build_report.json', encoding='utf-8'))
print('Split sizes:', report['split_sizes'])
print('Per source:', report['per_source'])
print('Leakage check:', report['leakage_check'])

In [ ]:
# Cell 5: Train with QLoRA (auto-resumes from the last checkpoint if one exists)
import subprocess

resume = None
checkpoints = sorted(OUTPUT_DIR.glob('checkpoint-*')) if OUTPUT_DIR.exists() else []
if checkpoints:
    resume = str(checkpoints[-1])
    print('Existing checkpoint found. Resuming from:', resume)
    print('(Delete ' + str(OUTPUT_DIR) + ' to train from scratch instead)')

c = CONFIG
cmd = [
    'python', str(SRC / 'train.py'),
    '--train_data', str(FINAL_DIR / 'train.jsonl'),
    '--eval_data', str(FINAL_DIR / 'val.jsonl'),
    '--output_dir', str(OUTPUT_DIR),
    '--model_name', c['model_name'],
    '--epochs', str(c['epochs']),
    '--batch_size', str(c['batch_size']),
    '--gradient_accumulation_steps', str(c['gradient_accumulation']),
    '--learning_rate', str(c['learning_rate']),
    '--max_seq_length', str(c['max_seq_length']),
    '--lora_r', str(c['lora_r']),
    '--lora_alpha', str(c['lora_alpha']),
    '--seed', str(c['seed']),
]
if resume:
    cmd += ['--resume_from_checkpoint', resume]

print('>>', ' '.join(cmd))
r = subprocess.run(cmd)
if r.returncode != 0:
    raise SystemExit('Training failed (exit ' + str(r.returncode) + '). Read the traceback above. Common fix: OOM -> set max_seq_length 1024 in Cell 1 and Restart & Run All.')

In [ ]:
# Cell 6: Evaluate fine-tuned vs baseline (same prompts, greedy decoding)
import json
import subprocess

cmd = [
    'python', str(SRC / 'evaluate.py'),
    '--model_dir', str(OUTPUT_DIR),
    '--eval_data', str(FINAL_DIR / 'test.jsonl'),
    '--baseline_model', CONFIG['model_name'],
    '--output', str(REPORTS_DIR / 'eval.json'),
]
print('>>', ' '.join(cmd))
r = subprocess.run(cmd)
if r.returncode != 0:
    raise SystemExit('Evaluation failed (exit ' + str(r.returncode) + ').')

ev = json.load(open(REPORTS_DIR / 'eval.json', encoding='utf-8'))
print()
print(f"{'Metric':<26}{'Baseline':<12}{'Fine-tuned':<12}")
print('-' * 50)
for key in ['rougeL', 'bleu', 'chrf', 'script_structure_score']:
    b = ev.get('baseline', {}).get(key, '-')
    f = ev.get('fine_tuned', {}).get(key, '-')
    print(f'{key:<26}{str(b):<12}{str(f):<12}')
print()
print('Copy these numbers into docs/BENCHMARKS.md')

In [ ]:
# Cell 7: Upload adapter + eval report to Hugging Face
from huggingface_hub import HfApi

api = HfApi(token=os.environ['HF_TOKEN'])
repo_id = os.environ['HF_USERNAME'] + '/' + os.environ['HF_REPO']
api.create_repo(repo_id=repo_id, exist_ok=True)
api.upload_folder(folder_path=str(OUTPUT_DIR), repo_id=repo_id)
if (REPORTS_DIR / 'eval.json').exists():
    api.upload_file(path_or_file=str(REPORTS_DIR / 'eval.json'), path_in_repo='eval_report.json', repo_id=repo_id)
print('Uploaded adapter + eval report to: https://huggingface.co/' + repo_id)

## After training

1. **Cell 6 results** -> fill the benchmark table in `docs/BENCHMARKS.md`
2. Adapter is on Hugging Face (Cell 7) - load with `peft.PeftModel.from_pretrained`
3. Deploy: `ollama create urdu-drama -f ollama/Modelfile`

## Troubleshooting

| Problem | Fix |
|---|---|
| OOM during training | Cell 1: `max_seq_length: 1024`, then Restart & Run All |
| Session cut mid-training | Re-run Cells 1-5; Cell 5 auto-resumes from the last checkpoint |
| 403 on model download | Use the NousResearch mirror (default) or get meta-llama access on HF |
| Token rejected | New token with Read+Write at huggingface.co/settings/tokens, re-run Cell 1 |
| P100 GPU | This script needs bfloat16 - switch to T4/A100 |

**Never put your token in a shared/exported notebook cell as plain text.**